# Лабораторная работа №5
## Вызов C-функции из Python четырьмя способами

**Вариант 5:** вычисление среднего, дисперсии и стандартного отклонения массива.

### Структура проекта
```
lab5-6/
├── c_core/           stats.h, stats.c, build.bat
├── python_ctypes/    bench_ctypes.py
├── python_cffi/      bench_cffi.py
├── python_capi/      statsmodule.c, setup.py
├── python_cython/    stats_cy.pyx, setup.py
└── benchmark/        (этот ноутбук)
```

## 0. Сборка

Перед запуском выполни в терминале из папки `lab5-6`:

```bat
REM 1. DLL
cd c_core && build.bat && cd ..

REM 2. CPython C API
cd python_capi && py -3.12 setup.py build_ext --inplace && cd ..

REM 3. Cython
cd python_cython && py -3.12 setup.py build_ext --inplace && cd ..
```

> Нужен GCC (MinGW): `winget install MSYS2.MSYS2` или `scoop install gcc`

In [ ]:
import sys, time, pathlib, statistics
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = pathlib.Path('..').resolve()
sys.path.insert(0, str(ROOT / 'python_ctypes'))
sys.path.insert(0, str(ROOT / 'python_cffi'))
sys.path.insert(0, str(ROOT / 'python_capi'))
sys.path.insert(0, str(ROOT / 'python_cython'))

print('ROOT:', ROOT)

## 1. Загрузка четырёх реализаций

In [ ]:
import bench_ctypes
import bench_cffi

try:
    import stats_capi
    HAS_CAPI = True
    print('stats_capi: OK')
except ImportError as e:
    HAS_CAPI = False
    print(f'stats_capi: недоступен ({e})')

try:
    import stats_cy
    HAS_CYTHON = True
    print('stats_cy: OK')
except ImportError as e:
    HAS_CYTHON = False
    print(f'stats_cy: недоступен ({e})')

print('ctypes: OK')
print('cffi:   OK')

## 2. Тестовые данные

100 000 массивов по 100 элементов, seed фиксирован.

In [ ]:
N_CALLS  = 100_000
N_RUNS   = 50
ARR_SIZE = 100

rng = np.random.default_rng(42)
test_data = [rng.standard_normal(ARR_SIZE).tolist() for _ in range(N_CALLS)]

print(f'Данные: {N_CALLS} массивов × {ARR_SIZE} элементов')

## 3. Функция замера

In [ ]:
def benchmark(fn, data, n_runs=N_RUNS, warmup=1000):
    for arr in data[:warmup]:
        fn(arr)
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        for arr in data:
            fn(arr)
        times.append(time.perf_counter() - t0)
    return times

## 4. Запуск бенчмарка

Каждый подход: 50 запусков × 100 000 вызовов.

In [ ]:
results = {}

print('ctypes ...', end=' ', flush=True)
results['ctypes'] = benchmark(bench_ctypes.compute_stats, test_data)
print('done')

print('cffi   ...', end=' ', flush=True)
results['cffi'] = benchmark(bench_cffi.compute_stats, test_data)
print('done')

if HAS_CAPI:
    print('C API  ...', end=' ', flush=True)
    results['C API'] = benchmark(lambda a: stats_capi.compute_stats(a), test_data)
    print('done')

if HAS_CYTHON:
    print('Cython ...', end=' ', flush=True)
    results['Cython'] = benchmark(stats_cy.stats, test_data)
    print('done')

print('\nГотово!')

## 5. Сводная таблица (секунды на 100 000 вызовов)

In [ ]:
rows = []
for name, times in results.items():
    rows.append({
        'Подход':  name,
        'Min':     round(min(times), 4),
        'Max':     round(max(times), 4),
        'Mean':    round(statistics.mean(times), 4),
        'Median':  round(statistics.median(times), 4),
        'Std':     round(statistics.stdev(times), 5),
    })

df = pd.DataFrame(rows).set_index('Подход')
df

## 6. Графики

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart: среднее время
ax = axes[0]
means = df['Mean']
bars = ax.bar(means.index, means.values, color=['steelblue', 'darkorange', 'seagreen', 'tomato'][:len(means)])
ax.bar_label(bars, fmt='%.4f', padding=3)
ax.set_ylabel('Среднее время (сек)')
ax.set_title('Среднее время 100 000 вызовов')
ax.grid(axis='y', alpha=0.3)

# Boxplot: распределение по 50 замерам
ax2 = axes[1]
ax2.boxplot(list(results.values()), labels=list(results.keys()), patch_artist=True)
ax2.set_ylabel('Время замера (сек)')
ax2.set_title('Boxplot (50 замеров)')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Выводы

| Критерий | Победитель | Примечание |
|---|---|---|
| Самый быстрый | Cython / C API | Минимум накладных расходов на Python↔C |
| Самый простой | ctypes | Только Python, компилятор не нужен |
| Наиболее удобный для сопровождения | cffi | Декларативный cdef, читается как заголовок |

**Причины различий:**
- `ctypes` и `cffi` работают через ABI-мост и выполняют маршалинг типов при каждом вызове.
- CPython C API и Cython работают напрямую внутри интерпретатора, маршалинг минимален.
- Cython дополнительно оптимизирует цикл на стороне Python до нативного кода.